# XPerfGPT weight binding

In [2]:
import os

os.environ['NCCL_DEBUG'] = 'WARN'
os.environ['SEED_MODELS_USE_XPERF_GENERATE'] = '1'

import seed_models # noqa

from verl.utils.fs import copy_local_path_from_hdfs
from verl.utils.distributed import initialize_global_process_group
from verl.utils.fsdp_utils import get_fsdp_wrap_policy

import torch
import torch.distributed
from transformers import AutoTokenizer, AutoModelForCausalLM

from torch.distributed.fsdp import FullyShardedDataParallel as FSDP
from torch.distributed.fsdp.api import ShardingStrategy, MixedPrecision

from torch.distributed.device_mesh import init_device_mesh


model_path = copy_local_path_from_hdfs(
    'hdfs://haruna/home/byte_data_seed/lf_lq/user/zhangchi.usc1992/seed_rl/models/P6.1_12B_32k_SFT29_Fix_RoPE_Base_hf')
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.padding_side = "left"

with torch.device('cpu'):
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    )

    config = model.config

model = model.cuda()


from verl.utils.model import create_random_mask, compute_position_id_with_mask
from mono_rl import DataProto

prompt = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"

chat = [{'role': 'user', 'content': prompt}]

sentences = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)

input_data = tokenizer(sentences, return_tensors='pt').to('cuda')

input_ids = input_data['input_ids']
attention_mask = input_data['attention_mask']
position_ids = compute_position_id_with_mask(attention_mask)

data = {'input_ids': input_ids, 'attention_mask': attention_mask, 'position_ids': position_ids}

output = model.generate(**data)

text_out = tokenizer.batch_decode(output, skip_special_tokens=True)

Unrecognized keys in `rope_scaling` for 'rope_type'='default': {'factor'}
You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

/home/tiger/.local/lib/python3.11/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable 

[WARNING][xperf.py:67] 11/06/2024 13:21:46 >> Unloading model from gpu memory, because xperf will allocate its own
[WARNING][xperf.py:68] 11/06/2024 13:21:46 >> It's recommanded that when using xperf_gpt mode, do not call .cuda() explicitly.


TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


[WARNING][xperf.py:89] 11/06/2024 13:22:00 >> Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[WARNING][xperf.py:53] 11/06/2024 13:22:00 >> Unrecognized generation kwargs in xperf_gpt generate: ['input_ids', 'attention_mask', 'position_ids']


jit_linear_gelu_linear:  False
[INFO][modeling_p6d.py:1346] 11/06/2024 13:22:40 >> Using xperf_gpt to generate
[WARNING][xperf.py:53] 11/06/2024 13:22:40 >> Unrecognized generation kwargs in xperf_gpt generate: ['position_ids']


In [4]:
print(text_out[0])

user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?assistant
In April, Natalia sold 48 clips.

In May, she sold half as many clips as in April. Half of 48 is \(48\div2 = 24\) clips.

To find out how many clips she sold altogether in April and May, we need to add the number of clips she sold in April and May.

The number of clips sold in April is 48 and the number of clips sold in May is 24.

The total number of clips sold is \(48+ 24=72\)

So Natalia sold 72 clips altogether in April and May.


In [9]:
print(model)

P6DenseForCausalLM(
  (model): P6DenseModel(
    (embed_tokens): Embedding(155136, 4608)
    (layers): ModuleList(
      (0-42): 43 x P6DenseDecoderLayer(
        (self_attn): P6DenseFlashAttention2(
          (q_proj): Linear(in_features=4608, out_features=4608, bias=False)
          (k_proj): Linear(in_features=4608, out_features=1536, bias=False)
          (v_proj): Linear(in_features=4608, out_features=1536, bias=False)
          (o_proj): Linear(in_features=4608, out_features=4608, bias=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
          (rotary_emb): P6DenseRotaryEmbedding()
        )
        (mlp): P6DenseMLP(
          (gate_proj): Linear(in_features=4608, out_features=16128, bias=False)
          (up_proj): Linear(in_features=4608, out_features=16128, bias=False)
          (down_proj): Linear(in_features=16128, out_features=4608, bias=False)
          (act_fn): SiLU()
          (dropout): Dropout(p=0.2, inplace=False)
        )
        (input_layernorm): 

In [41]:
print(model.config)

P6DenseConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "/tmp/ce63db5377da52f207677d5324d2d469/P6.1_12B_32k_SFT29_Fix_RoPE_Base_hf",
  "architectures": [
    "P6DenseForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "eos_token_id": 2,
  "hidden_act": "silu",
  "hidden_size": 4608,
  "initializer_range": 0.01613743060919757,
  "intermediate_size": 16128,
  "layer_norm_eps": null,
  "max_position_embeddings": 32768,
  "mlp_bias": false,
  "model_type": "seed_p6dense",
  "num_attention_heads": 36,
  "num_hidden_layers": 43,
  "num_key_value_heads": 12,
  "resid_pdrop": 0.2,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 1,
    "rope_type": "default"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "bfloat16",
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 155136
}



In [5]:
torch.testing.assert_close(model._xperf_optimized_model.module.wte_weight.data, model.model.embed_tokens.weight.cuda())

In [7]:
torch.testing.assert_close(model._xperf_optimized_model.module.lm_head_weight.data, model.lm_head.weight.cuda())

In [8]:
torch.testing.assert_close(model._xperf_optimized_model.module.layernorm_weight.data, model.model.norm.weight.cuda().unsqueeze(0))

In [11]:
torch.testing.assert_close(model._xperf_optimized_model.module.layers_weight[0][0].data, model.model.layers[0].input_layernorm.weight.cuda().unsqueeze(0))

In [12]:
torch.testing.assert_close(model._xperf_optimized_model.module.layers_weight[0][5].data, model.model.layers[0].post_attention_layernorm.weight.cuda().unsqueeze(0))

In [31]:
torch.testing.assert_close(model._xperf_optimized_model.module.layers_weight[0][3].data, model.model.layers[0].self_attn.o_proj.weight.cuda().view(4608, 12, 3, 128).transpose(1, 2).reshape(4608, 4608))

In [38]:
q = model.model.layers[0].self_attn.q_proj.weight.cuda().view(12, 3, 128, 4608).transpose(0, 1).reshape(4608, 4608)
k = model.model.layers[0].self_attn.k_proj.weight.cuda()
v = model.model.layers[0].self_attn.v_proj.weight.cuda()
qkv = torch.cat((q, k, v), dim=0)

In [40]:
torch.testing.assert_close(model._xperf_optimized_model.module.layers_weight[0][1].data, qkv)